# Cluster Setup Guide

## Prerequisites
- Databricks workspace (Free Edition or higher)
- INSEE API key (see Step 3)
- GitHub account

## Step 1 — Fork and link GitHub repo
1. Fork this repo to your own GitHub account
2. Databricks → User Settings → Linked accounts → GitHub
3. Generate a GitHub Classic token with `repo` and `workflow` scopes
4. Connect and clone your forked repo into your Workspace
5. Rename `.env.example` to `.env` and fill in your credentials

## Step 2 — Install package on cluster

In [0]:
import subprocess
result = subprocess.run(
    ["pip", "install", "-e", f"/Workspace/Users/YOUR_USER/insee-sirene-monitor"],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

In [0]:
# restart kernel to get access to project packages
dbutils.library.restartPython() 

## Step 3a — Configure Databricks credentials

In [0]:
from dotenv import load_dotenv
import os

DATABRICKS_USER='YOUR_DATABRICKS_USER'

load_dotenv(f"/Workspace/Users/{DATABRICKS_USER}/insee-sirene-monitor/.env")

DATABRICKS_HOST = os.environ["DATABRICKS_HOST"]
DATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]
ALERT_EMAIL = os.environ["ALERT_EMAIL"]
INSEE_API_KEY = os.environ["INSEE_API_KEY"]

In [0]:
import requests

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

# Create scopes
requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/scopes/create", headers=headers,
    json={"scope": "databricks-credentials"})
requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/scopes/create", headers=headers,
    json={"scope": "insee-sirene-monitor-api-credentials"})

# Add secrets
for key, value in [
    ("host", DATABRICKS_HOST),
    ("token", DATABRICKS_TOKEN),
    ("alert-email", ALERT_EMAIL),
    ("project-path",f"/Workspace/Users/{DATABRICKS_USER}/insee-sirene-monitor")
]:
    requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/put", headers=headers,
        json={"scope": "databricks-credentials", "key": key, "string_value": value})

requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/put", headers=headers,
    json={"scope": "insee-sirene-monitor-api-credentials", "key": "api_key", "string_value": INSEE_API_KEY})

print("Credentials configured")

## Step 3b — Configure INSEE API credentials

Get your INSEE API key at: https://portail-api.insee.fr
- Create a new "simple" application (not "backend to backend")
- Go to "Souscriptions" tab
- Subscribe to "Sirene V3" API
- Got to your app space, "Souscriptions" tab, select the API, and copy the API Key displayed on the right part of the screen

In [0]:
# Create scope
requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/scopes/create", headers=headers,
    json={"scope": "insee-sirene-monitor-api-credentials"})

# Add API key
requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/put", headers=headers,
    json={
        "scope": "insee-sirene-monitor-api-credentials",
        "key": "api_key",
        "string_value":INSEE_API_KEY
    })

print("INSEE credentials configured")

## Step 4 — Verify INSEE API connection

In [0]:
from utils.storage import get_insee_api_key

api_key = get_insee_api_key(dbutils)

response = requests.get(
    "https://api.insee.fr/api-sirene/3.11/siren/309634954",
    headers={"X-INSEE-Api-Key-Integration": api_key}
)
print("INSEE API OK:", response.status_code == 200)

Expected output: `INSEE API OK: True`

##Step 5 — Install dbt dependencies

In [0]:
import subprocess
from utils.config import DBT_PROJECT_DIR

result = subprocess.run(
    ["dbt", "deps", "--project-dir", DBT_PROJECT_DIR, "--profiles-dir", DBT_PROJECT_DIR],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

Expected output:
```
Running with dbt=1.11.6  
Updating lock file in file path: /Workspace/Users/...  
Installing dbt-labs/dbt_utils  
Installed from version 1.3.3  
Up to date!
```

##Step 6 — Deploy Databricks jobs and store job IDs

In [0]:
import yaml
from utils.config import DATABRICKS_USER

base_path = f"/Workspace/Users/{DATABRICKS_USER}/insee-sirene-monitor"

def deploy_job(yaml_path: str) -> int:
    with open(yaml_path) as f:
        content = f.read()
    
    # inject project path
    content = content.replace("{{project_path}}", base_path)
    settings = yaml.safe_load(content)
    
    # replace relative paths with absolute paths
    for task in settings.get("tasks", []):
        if "spark_python_task" in task:
            task["spark_python_task"]["python_file"] = \
                f"{base_path}/{task['spark_python_task']['python_file']}"

    # check if job already exists
    existing_jobs = requests.get(
        f"{DATABRICKS_HOST}/api/2.1/jobs/list",
        headers=headers
    ).json().get("jobs", [])
    
    existing = [j for j in existing_jobs if j["settings"]["name"] == settings["name"]]
    
    if existing:
        job_id = existing[0]["job_id"]
        requests.post(
            f"{DATABRICKS_HOST}/api/2.1/jobs/reset",
            headers=headers,
            json={"job_id": job_id, "new_settings": settings}
        )
        print(f"Updated job '{settings['name']}' (id: {job_id})")
    else:
        response = requests.post(
            f"{DATABRICKS_HOST}/api/2.1/jobs/create",
            headers=headers,
            json=settings
        )
        job_id = response.json()["job_id"]
        print(f"Created job '{settings['name']}' (id: {job_id})")
    
    return job_id

bronze_silver_job_id = deploy_job(f"{base_path}/jobs/bronze_silver.yml")
silver_gold_job_id = deploy_job(f"{base_path}/jobs/silver_gold.yml")
reset_pipeline_job_id = deploy_job(f"{base_path}/jobs/reset_pipeline.yml")
ingestion_bronze_job_id = deploy_job(f"{base_path}/jobs/ingestion_bronze.yml")

for key, value in [
    ("bronze-silver-job-id", str(bronze_silver_job_id)),
    ("silver-gold-job-id", str(silver_gold_job_id)),
    ("reset-pipeline-job-id", str(reset_pipeline_job_id)),
    ("ingestion-bronze-job-id", str(ingestion_bronze_job_id)),
]:
    requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/put", headers=headers,
        json={"scope": "databricks-credentials", "key": key, "string_value": value})

print(f"\nJob IDs stored in vault:")
print(f"  ingestion-bronze-job-id : {ingestion_bronze_job_id}")
print(f"  bronze-silver-job-id    : {bronze_silver_job_id}")
print(f"  silver-gold-job-id      : {silver_gold_job_id}")
print(f"  reset-pipeline-job-id   : {reset_pipeline_job_id}")

In [0]:
from nacl import encoding, public
import base64

GITHUB_TOKEN = os.environ["GITHUB_TOKEN"]
GITHUB_REPO = os.environ["GITHUB_REPO"]

github_headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

# get repository public key
pub_key_response = requests.get(
    f"https://api.github.com/repos/{GITHUB_REPO}/actions/secrets/public-key",
    headers=github_headers
).json()

pub_key = pub_key_response["key"]
pub_key_id = pub_key_response["key_id"]

def encrypt_secret(public_key: str, secret_value: str) -> str:
    pk = public.PublicKey(public_key.encode("utf-8"), encoding.Base64Encoder())
    sealed_box = public.SealedBox(pk)
    encrypted = sealed_box.encrypt(secret_value.encode("utf-8"))
    return base64.b64encode(encrypted).decode("utf-8")

for name, value in [
    ("DATABRICKS_TOKEN", DATABRICKS_TOKEN),
    ("DATABRICKS_HOST", DATABRICKS_HOST),
    ("SILVER_GOLD_JOB_ID", str(silver_gold_job_id)),
    ("RESET_PIPELINE_JOB_ID", str(reset_pipeline_job_id)),
]:
    response = requests.put(
        f"https://api.github.com/repos/{GITHUB_REPO}/actions/secrets/{name}",
        headers=github_headers,
        json={
            "encrypted_value": encrypt_secret(pub_key, value),
            "key_id": pub_key_id
        }
    )
    print(f"Secret {name}: {response.status_code}")